In [1]:
# Import module constrained_likelihood_surrogates
#
import sys
sys.path.append("../src/")
from constrained_likelihood_surrogates import *

In [2]:
import pickle
with open("../data/dataBeagle1000.pkl","rb") as f:
    dictR=pickle.load(f)
with open("../data/countBeagle1000.pkl","rb") as f:
    dictC=pickle.load(f)
with open("../data/burstinessBeagle1000.pkl","rb") as f:
    dictB=pickle.load(f)

In [3]:
print(dictR["sand"])
print(dictB["sand"])
print(dictC["sand"])

[6481, 212, 4249, 2436, 5660, 8135, 39, 63, 43, 123, 55, 86, 185, 50, 117, 9, 447, 1721, 3443, 17, 81, 631, 1442, 348, 738, 3053, 2457, 546, 116, 60, 662, 14376, 1452, 14065, 3918, 6830, 432, 195, 1719, 4696, 4240, 9807, 1859, 20872, 5318, 603, 11258, 1993, 2776, 1548, 980, 14, 101, 12, 217, 52, 592, 38, 143, 2696, 2612, 40, 2756, 7681, 718, 15403, 3415, 1106, 166, 69, 1234, 497, 29, 1921, 32, 1907, 117, 3041, 599, 8017, 509]
1.5543509732327725
81


In [4]:
if 1:#All 1000 most frequent words
    word_list = list(dictR.keys())
else:#A selection of words
    count_word_burstiness_list = \
    [\
     [16882, 'the', '0.9127467399523432'],#Two most common
     [9414, 'of', '0.9674115529397951'],
     [1379, 'this', '1.064848821764014'],#Selected from 1000-1500
     [1165, 'we', '1.6621835656808999'],
     [1073, 'one', '1.0208032765847226'],
     [1040, 'they', '1.54209675726809'],
     [364, 'many', '1.0228254008708726'],#Selected from 300-400
     [337, 'country', '1.5051897284815428'],
     [308, 'no', '1.050665453951139'],
     [303, 'species', '2.441175487780599'],
     [114, 'remarkable', '0.9927927917468405'],#Selected from 100-120
     [110, 'river', '2.3233345890045296'],
     [109, 'fine', '1.0522972390432823'],
     [106, 'snow', '2.645420200799017'],
     [106, 'half', '0.8911069745854838'],
     [105, 'others', '0.9219188231440477'],
     [30, 'watch', '1.211488472023234'],#Selected from 30-30
     [30, 'use', '0.872154621277274'],
     [30, 'sure', '0.9883558728905495'],
     [30, 'support', '0.9826977807784691'],
     [30, 'successive', '1.0627796806684702'],
     [30, 'strait', '2.650902808433499'],
     [30, 'showed', '1.564223055214392'],
     [30, 'sandstone', '1.6802010085265553'],
     [30, 'riding', '1.4305587299861329'],
     [30, 'regions', '1.0539322952394314'],
     [30, 'reason', '0.9422151245434213'],
     [30, 'purpose', '0.8952588262864375'],
     [30, 'prey', '1.181826323921316'],
     [30, 'peru', '1.7156224585294277'],
     [30, 'noticed', '0.8623287204103504'],
     [30, 'la', '1.4056755770779976'],
     [30, 'feeling', '1.325833949151611'],
     [30, 'completely', '0.9489851199369673'],
     [30, 'collected', '1.0220016219824828'],
     [30, 'cliff', '1.5203536471551793'],
     [30, 'chapter', '0.5913085349511107'],
     [30, 'brown', '1.2395809322717672'],
     [30, 'brazil', '1.3865345995612905'],
     [30, 'australia', '1.3052698380835763'],
     [30, 'andes', '2.3607365307046275'],
     [30, 'ancient', '1.213762202865566'],
     [30, 'absolutely', '0.8476159276700899']]
    word_list = [word for [count, word, burstiness] in count_word_burstiness_list]
    word_list = word_list[::-1]

In [5]:
# Determining REJECTION RATE from constrained and typical surrogates for WORDS DATA (selected waiting times for words from Darwin's book on his voyage in the Beagle):
# 
# Use several types of statistics as test statistics
# 
# Considering constrained and typical surrogates
# 
# 
# Model-free statistics:
free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth]
# free_stat_list = free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth, geom_mean, harm_mean, rang, log_rang, iq_rang, log_iq_rang]
num_free_stats = len(free_stat_list)
# Model-dependent statistics (dependent on one model - this characterisation is an excuse so leave out the likelihood ratio):
# depe_stat_list = [ks_stat, ad_stat, cvm_stat]
depe_stat_list = [ks_stat, kuiper_stat, ad_stat, cvm_stat, zk_stat, za_stat, zc_stat]
num_depe_stats = len(depe_stat_list)

model = 'expon'
model_list = [model]
num_models = len(model_list)

test_type = 'rejection'
num_words = len(word_list)
print('Considering ' + str(num_words) + ' words')
for i_word in range(num_words):
    start_time = timer()
    word = word_list[i_word]
    val_seq_full = dictR[word]
    burstiness = dictB[word]
    count = dictC[word]

    # # Find lower cutoffs by minimising KS distance
    # # ks_method = 'ave'
    # ks_method = 'sup'
    # N_full = len(val_seq_full)
    # print('Word = ' + word + ', N_full = ' + str(N_full) + ', min. = ' + str(min(val_seq_full)) + ', max. = ' + str(max(val_seq_full)) + ', mean = ' + str(np.mean(val_seq_full)))
    # lower_cutoff, ks, lower_cutoff_hat_seq, ks_seq = fit_lower_cutoff(val_seq_full, model=model, continuous=False, ks_method=ks_method)
    
    # lower_cutoff = 4.5
    lower_cutoff = 9.5#Considered in main text
    # lower_cutoff = 19.5
    
    val_seq = [val for val in val_seq_full if (val >= lower_cutoff)]
    
    N = len(val_seq)
    num_trans = N*(int(np.ceil(np.log2(N*1024))))
    
    num_surr = 999
    num_tests = 10**0
    
    print('N = ' + str(N) + ', lower cut-off = ' + str(lower_cutoff))
    
    print(str(num_tests) + ' tests, each with ' + str(num_surr) + ' constrained, typical surrogates, each using ' + str(num_trans) + ' transitions')
    
    num_methods = 2#Based on constrained, typical
    
    # with warnings.catch_warnings():
    #     warnings.filterwarnings('error', message='invalid value encountered in subtract')
    #     warnings.filterwarnings('error', message='divide by zero encountered in log')
    #     warnings.filterwarnings('error', message='divide by zero encountered in double_scalars')
    #     warnings.filterwarnings('error', message='invalid value encountered in true_divide')
    
    
    folder_str = './results/hyp-test/words/'
    
    # save_str_0 = 'quant_free-depe' + '_' + test_type + '_xmin-' + str(lower_cutoff) + '_N-' + str(N) + '_ntra-' + str(num_trans) + '_nsur-' + str(num_surr) + '_ntes-' + str(num_tests) + '_ntyp-' + str(num_methods) + '_nmod-' + str(num_models) + '_nfs-' + str(num_free_stats) + '_nds-' + str(num_depe_stats)
    save_str_0 = 'quant_free-depe' + '_' + test_type + '_xmin-' + str(lower_cutoff) + '_nsur-' + str(num_surr) + '_ntes-' + str(num_tests) + '_ntyp-' + str(num_methods) + '_mod-' + str(model) + '_nfs-' + str(num_free_stats) + '_nds-' + str(num_depe_stats)
    process_str = '_wait-seq-' + word
    save_str = save_str_0 + process_str
    
    if exists(folder_str + save_str + '.json'):
        print('Skipping pre-existing file: ' + save_str + '.json')
        continue
    
    free_quantile_list_list = np.full((num_methods, num_models, num_tests, num_free_stats), np.nan)#Array of nans
    depe_quantile_list_list = np.full((num_methods, num_models, num_tests, num_depe_stats), np.nan)#Array of nans

    for i_test in range(num_tests):
        #Generate time series and calculate ks distance:
        lower_cutoff_hat = lower_cutoff
        free_stat_val_list = [stat(val_seq) for stat in free_stat_list]
        for i_model in range(num_models):
            model = model_list[i_model]
            param_hat_m, lp_seq_m = fit_model(val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)#m for model
            val_seq_sorted = sorted(val_seq)
            emp_cdf_with_rep = rankdata(val_seq_sorted, method='max')/len(val_seq_sorted)
            cdf_fun = return_cdf_func(param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
            exp_cdf_with_rep = cdf_fun(val_seq_sorted)
            depe_stat_val_list = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
            
            c_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Constrained
            t_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Typical
            
            c_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Constrained
            t_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Typical
            
            surr_m_val_seq = val_seq
            surr_m_val_seq_list = []
            for i_surr in range(num_surr):
                method = 'constrained'
                surr_m_val_seq = gen_surrogate(surr_m_val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)
                random.shuffle(surr_m_val_seq)
                surr_m_val_seq_list = surr_m_val_seq_list + [surr_m_val_seq]

            for i_surr in range(num_surr):
                method = 'constrained'
                surr_m_val_seq = surr_m_val_seq_list[i_surr]
                surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                c_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                c_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m

            for i_surr in range(num_surr):
                method = 'typical'
                surr_m_val_seq = gen_surrogate(val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)
                surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                t_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                t_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m

            for i_free_stat in range(num_free_stats):
                #Calculate values of discriminating statistic (model-free statistics) for observed sequence and surrogate sequences:
                obs_stat = free_stat_val_list[i_free_stat]
                abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                if (abs_obs_stat == np.inf):
                    abs_obs_stat = 0
                obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                for i_method in range(num_methods):
                    if (i_method == 0):#Constrained surrogates
                        stat_val_surr_m_list = c_surr_m_free_list_list[:, i_free_stat]
                    elif (i_method == 1):#Typical surrogates
                        stat_val_surr_m_list = t_surr_m_free_list_list[:, i_free_stat]
                    stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                    #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                    rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                    rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                    r = random.randint(rankMin, rankMax)
                    q = (r - 0.5)/(num_surr + 1)
                    free_quantile_list_list[i_method, i_model, i_test, i_free_stat] = q
                    if (np.isnan(q)):
                        raise Exception('Calculated quantile q is not a number.')

            for i_depe_stat in range(num_depe_stats):
                #Calculate values of discriminating statistic (model-dependent statistics) for observed sequence and surrogate sequences:
                obs_stat = depe_stat_val_list[i_depe_stat]
                abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                if (abs_obs_stat == np.inf):
                    abs_obs_stat = 0
                obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                for i_method in range(num_methods):
                    if (i_method == 0):#Constrained surrogates
                        stat_val_surr_m_list = c_surr_m_depe_list_list[:, i_depe_stat]
                    elif (i_method == 1):#Typical surrogates
                        stat_val_surr_m_list = t_surr_m_depe_list_list[:, i_depe_stat]
                    stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                    #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                    rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                    rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                    r = random.randint(rankMin, rankMax)
                    q = (r - 0.5)/(num_surr + 1)
                    depe_quantile_list_list[i_method, i_model, i_test, i_depe_stat] = q
                    if (np.isnan(q)):
                        raise Exception('Calculated quantile q is not a number.')

    end_time = timer()
    total_time = end_time - start_time
    print('Word ' + word + ' with N=' + str(N) + ' took ' + str(total_time) + 'sec.') # Time in seconds, e.g. 5.38091952400282

    free_stat_name_list = [stat.__name__ for stat in free_stat_list]
    depe_stat_name_list = [stat.__name__ for stat in depe_stat_list]

    save_dict = {'free_quantile_list_list':free_quantile_list_list.tolist(),
                 'depe_quantile_list_list':depe_quantile_list_list.tolist(),
                 'test_type':test_type,
                 'lower_cutoff':lower_cutoff,
                 'N':N,
                 'num_trans':num_trans,
                 'num_surr':num_surr,
                 'num_tests':num_tests,
                 'num_methods':num_methods,
                 'process_str':process_str,
                 'model_list':model_list,
                 'model':model,
                 'save_str':save_str,
                 'total_time':total_time,
                 'free_stat_name_list':free_stat_name_list,
                 'depe_stat_name_list':depe_stat_name_list,
                 'val_seq':val_seq,
                 'val_seq_full':val_seq_full,
                 'word':word,
                 'burstiness':burstiness,
                 'count':count,
    }
    
    save(folder_str + save_str, save_dict)

Considering 43 words
N = 30, lower cut-off = 9.5
1 tests, each with 9 constrained, typical surrogates, each using 450 transitions
Word absolutely with N=30 took 0.1994847000005393sec.
N = 30, lower cut-off = 9.5
1 tests, each with 9 constrained, typical surrogates, each using 450 transitions
Word ancient with N=30 took 0.21434210000006715sec.
N = 30, lower cut-off = 9.5
1 tests, each with 9 constrained, typical surrogates, each using 450 transitions
Word andes with N=30 took 0.15692149999995308sec.
N = 30, lower cut-off = 9.5
1 tests, each with 9 constrained, typical surrogates, each using 450 transitions
Word australia with N=30 took 0.1491157000000385sec.
N = 30, lower cut-off = 9.5
1 tests, each with 9 constrained, typical surrogates, each using 450 transitions
Word brazil with N=30 took 0.14178770000035001sec.
N = 30, lower cut-off = 9.5
1 tests, each with 9 constrained, typical surrogates, each using 450 transitions
Word brown with N=30 took 0.15491380000094068sec.
N = 30, lower c